# Preparación reproducible de SpaPhish V5

Este notebook ejecuta el script de preparación, presenta sus controles y verifica el corpus binario inicial. No incorpora SpearPhishMX, no cambia etiquetas y no entrena modelos.

In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists():
    ROOT = ROOT.parent.parent
SCRIPT = ROOT / 'notebooks/preparacion-datos/prepare_spaphish_v5.py'
OUTPUT = ROOT / 'notebooks/data/processed/spaphish_v5'
SOURCE = ROOT / 'notebooks/data/spanish-dataset/SpanishDataset.csv'
SOURCE_SHA256 = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
SOURCE_SHA256

'656b2245d58da72d640680e5c2a168673a130b38607f2a427c773bbb167e995e'

## Ejecución

El script comprueba el hash de la fuente antes de escribir resultados. La fuente se abre únicamente para lectura.

In [2]:
result = subprocess.run([sys.executable, str(SCRIPT)], cwd=ROOT, check=True, capture_output=True, text=True)
print(result.stdout)

{
  "records": {
    "original": 1395,
    "conserved": 1395,
    "excluded": 0,
    "exclusion_reasons": {},
    "missing_subject_conserved": 3,
    "missing_body": 0,
    "encoding_warning_records_conserved": 16
  },
  "partition": {
    "train": {
      "records": 977,
      "proportion": 0.700358,
      "target_0": 466,
      "target_1": 511,
      "groups": 905
    },
    "validation": {
      "records": 208,
      "proportion": 0.149104,
      "target_0": 99,
      "target_1": 109,
      "groups": 197
    },
    "test": {
      "records": 210,
      "proportion": 0.150538,
      "target_0": 99,
      "target_1": 111,
      "groups": 199
    }
  }
}



In [3]:
manifest = json.loads((OUTPUT / 'manifest.json').read_text(encoding='utf-8'))
corpus = pd.read_csv(OUTPUT / 'spaphish_v5_corpus.csv')
groups = pd.read_csv(OUTPUT / 'group_review.csv')
candidates = pd.read_csv(OUTPUT / 'near_duplicate_candidates.csv')
pd.DataFrame(manifest['partition']['actual']).T

,groups,proportion,records,target_0,target_1
test,199.0,0.150538,210.0,99.0,111.0
train,905.0,0.700358,977.0,466.0,511.0
validation,197.0,0.149104,208.0,99.0,109.0


## Duplicados y grupos

Los grupos se forman con plantillas normalizadas exactas o con similitud coseno de caracteres >= 0,95 **y** asunto normalizado idéntico. Los demás candidatos permanecen explícitamente pendientes.

In [4]:
display(pd.Series(manifest['duplicates_and_groups'], name='valor').to_frame())
display(groups[['group_size', 'target', 'rules', 'subject_variants', 'review_status']].head(10))
candidates.groupby(['decision', 'cross_split']).size().rename('pares').to_frame()

,valor
exact_normalized_groups,0
exact_template_groups,26
label_conflict_groups,0
max_group_size,10
multi_record_groups,47
near_candidate_pairs_cosine_ge_0.90,486
near_grouping_pairs_cosine_ge_0.95_same_subject,121
near_pairs_cosine_ge_0.95,154
pending_candidate_pairs,325
pending_candidate_pairs_cross_split,142


,group_size,target,rules,subject_variants,review_status
0,4,1,cosine_ge_0.95_same_subject|exact_template,1,accepted_conservative_group
1,2,1,cosine_ge_0.95_same_subject,1,accepted_conservative_group
2,2,1,exact_template,1,accepted_conservative_group
3,2,1,cosine_ge_0.95_same_subject,1,accepted_conservative_group
4,2,1,cosine_ge_0.95_same_subject,1,accepted_conservative_group
5,4,1,cosine_ge_0.95_same_subject,1,accepted_conservative_group
6,2,1,cosine_ge_0.95_same_subject,1,accepted_conservative_group
7,3,1,exact_template,3,accepted_conservative_group
8,2,1,exact_template,1,accepted_conservative_group
9,7,1,cosine_ge_0.95_same_subject|exact_template,1,accepted_conservative_group


pares
decision                 cross_split       
grouped                  False          161
pending_semantic_review  False           10
                         True            16
pending_threshold_review False          173
                         True           126

## Controles de integridad

Estas aserciones comprueban la fuente, el esquema, las etiquetas, la exclusión de SpearPhishMX y la separación estricta por grupos.

In [5]:
assert SOURCE_SHA256 == manifest['dataset']['source_sha256']
assert len(corpus) == 1395
assert corpus['record_id'].is_unique
assert set(corpus['target']) == {0, 1}
assert set(corpus['language']) == {'es'}
assert set(corpus['source_dataset']) == {'spaphish_v5'}
assert not corpus['text'].fillna('').str.strip().eq('').any()
assert not corpus.groupby('group_id')['split'].nunique().gt(1).any()
assert corpus.groupby('group_id')['target'].nunique().max() == 1
assert manifest['records']['excluded'] == 0
print('Controles superados: corpus íntegro, etiquetas preservadas y grupos sin solapamiento.')

Controles superados: corpus íntegro, etiquetas preservadas y grupos sin solapamiento.
